# 0-14 Yaş Oranı ve 100 Bin Kişi Başına Hastane Yatak Sayısı Tahmin Notebook'u

Bu notebook iki hedef için baştan sona çalışır:

1. `yas_0_14`
2. `yatak_oran` = 100 bin kişi başına düşen toplam hastane yatak sayısı

`yatak_oran`, `model_giris.csv` içindeki `yatak` ve `nufus` sütunlarından şu şekilde hesaplanır:

```python
yatak_oran = yatak / nufus * 100000
```

Dosyalar notebook ile aynı klasördeyse hiçbir path değiştirmen gerekmez:

- `model_giris.csv`
- `2025nufus-yas.csv`


In [2]:
# ============================================================
# 1) KÜTÜPHANELER
# ============================================================

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    BaggingRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except Exception:
    HAS_CAT = False

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    HAS_HW = True
except Exception:
    HAS_HW = False

try:
    from statsmodels.tsa.arima.model import ARIMA
    HAS_ARIMA = True
except Exception:
    HAS_ARIMA = False

print("Çalışma dizini:", os.getcwd())
print("Klasördeki dosyalar:")
print(os.listdir())

Çalışma dizini: C:\Users\Bedirhan\Desktop\ARSEK\saglik-personel-yer-secimi\yenicalismatopsis
Klasördeki dosyalar:
['.ipynb_checkpoints', '06_topsis_veri_hazirla.ipynb', '06_topsis_veri_hazirla.py', '07b_yas_yatak_genis_feature.ipynb', '07b_yas_yatak_genis_feature.py', '07c_yas_0_14_hastane_tahmin.ipynb', '07d_yas_0_14_yatak_oran_tahmin.ipynb', '07_topsis_model_egitim.ipynb', '07_topsis_model_egitim.py', '08_topsis_ensemble_gelecek_tahmin.py', '1000kisibasinahekim', '1000kisibasinahekim.csv', '100binkisibasinahastaneyatak', '100binkisibasinahastaneyatak.csv', '2025nufus-yas.csv', 'catboost_info', 'model_giris.csv', 'topsis_en_iyi_modeller.csv', 'topsis_en_iyi_yas_yatak_genis.csv', 'topsis_hedef_yillar.csv', 'topsis_metrikleri_detay.csv', 'topsis_metrikleri_genel.csv', 'topsis_metrikleri_yas_yatak_genis.csv', 'topsis_tahminler_fold5.csv', 'topsis_veri.csv', 'yasgruplari', 'yas_0_14_hastane_en_iyi_modeller.csv', 'yas_0_14_hastane_fold_sonuclari.csv', 'yas_0_14_hastane_model_ozet.csv', 'ya

In [3]:
# ============================================================
# 2) DOSYA YOLLARI
# ============================================================

# Dosyalar notebook ile aynı klasördeyse böyle kalmalı.
MODEL_GIRIS_PATH = "model_giris.csv"
YAS_PATH = "2025nufus-yas.csv"

if not os.path.exists(MODEL_GIRIS_PATH):
    raise FileNotFoundError(f"{MODEL_GIRIS_PATH} bulunamadı. CSV dosyasını notebook ile aynı klasöre koy.")

if not os.path.exists(YAS_PATH):
    raise FileNotFoundError(f"{YAS_PATH} bulunamadı. CSV dosyasını notebook ile aynı klasöre koy.")

print("Model giriş dosyası bulundu:", MODEL_GIRIS_PATH)
print("Yaş dosyası bulundu:", YAS_PATH)

Model giriş dosyası bulundu: model_giris.csv
Yaş dosyası bulundu: 2025nufus-yas.csv


In [4]:
# ============================================================
# 3) YARDIMCI FONKSİYONLAR
# ============================================================

TR_MAP = str.maketrans({
    "İ": "i", "I": "i", "ı": "i",
    "Ş": "s", "ş": "s",
    "Ğ": "g", "ğ": "g",
    "Ü": "u", "ü": "u",
    "Ö": "o", "ö": "o",
    "Ç": "c", "ç": "c",
})

def normalize_city_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"-\d+$", "", x)
    x = x.translate(TR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def clean_numeric(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    s = s.replace(" ", "")
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    if s in ["", "-", ".", "-."]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    if len(np.unique(y_true)) <= 1:
        r2 = np.nan
    else:
        r2 = r2_score(y_true, y_pred)

    eps = 1e-9
    mape = np.mean(
        np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))
    ) * 100

    accuracy = max(0, 100 - mape)

    return {
        "Accuracy": accuracy,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "ErrorRate": mape
    }

def show_target_diagnostics(df, target):
    print("\n" + "="*80)
    print(f"HEDEF KONTROLÜ: {target}")
    print("="*80)

    print("\nGenel özet:")
    print(df[target].describe())

    print("\nYıl bazında varyans kontrolü:")
    diag = (
        df.groupby("yıl")[target]
        .agg(["count", "nunique", "mean", "std", "min", "max"])
        .round(4)
    )
    display(diag.tail(15))

    low_var = diag[(diag["std"].fillna(0) < 1e-6) | (diag["nunique"] <= 1)]

    if len(low_var) > 0:
        print("\nUYARI: Bazı yıllarda hedef değişken tek değerli/çok düşük varyanslı görünüyor.")
        display(low_var)
    else:
        print("\nSorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.")

In [5]:
# ============================================================
# 4) MODEL GİRİŞ VERİSİNİ OKU
# ============================================================

df_main = pd.read_csv(MODEL_GIRIS_PATH)
df_main.columns = [c.strip() for c in df_main.columns]

# yatak_oran için yatak ve nufus zorunlu
required_cols = ["il_adı", "yıl", "yatak", "nufus"]
missing = [c for c in required_cols if c not in df_main.columns]

if missing:
    raise ValueError(f"model_giris.csv içinde eksik sütunlar var: {missing}")

df_main["il_key"] = df_main["il_adı"].apply(normalize_city_name)
df_main["yıl"] = df_main["yıl"].astype(int)

for c in df_main.columns:
    if c not in ["il_adı", "il_key"]:
        df_main[c] = pd.to_numeric(df_main[c], errors="coerce")

print(df_main.shape)
display(df_main.head())
print("Yıl aralığı:", df_main["yıl"].min(), "-", df_main["yıl"].max())
print("İl sayısı:", df_main["il_key"].nunique())


(1863, 12)


,il_adı,yıl,hastane,yatak,nufus,asistan,diger,ebe,hemsire,pratisyen,uzman,il_key
0,Adana,2002,18.0,4266.0,1916637.0,646.0,1182.0,1110.0,1927.0,826.0,1227.0,adana
1,Adana,2003,21.0,4382.0,1933428.0,794.0,1414.0,1147.0,2118.0,844.0,1439.0,adana
2,Adana,2004,21.0,4207.0,1951142.0,856.0,1469.0,1167.0,2157.0,892.0,1497.0,adana
3,Adana,2005,22.0,4773.0,1969512.0,869.0,1540.0,1186.0,2223.0,979.0,1555.0,adana
4,Adana,2006,21.0,4973.0,1988277.0,409.0,1683.0,1258.0,2235.0,938.0,1364.0,adana


Yıl aralığı: 2002 - 2024
İl sayısı: 81


In [6]:
# ============================================================
# 5) YAŞ DOSYASINI PARSE ET
# ============================================================

def parse_age_file(path):
    rows = []
    current_year = None

    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split("|")
            first = parts[0].strip() if len(parts) > 0 else ""
            second = parts[1].strip() if len(parts) > 1 else ""

            if first.isdigit():
                current_year = int(first)
                city_raw = second
                values = parts[2:]
            elif current_year is not None and second:
                city_raw = second
                values = parts[2:]
            else:
                continue

            if len(values) < 9:
                continue

            rows.append({
                "yıl": current_year,
                "il_raw": city_raw,
                "il_key": normalize_city_name(city_raw),
                "yas_0_4": clean_numeric(values[0]),
                "yas_10_14": clean_numeric(values[1]),
                "yas_5_9": clean_numeric(values[2]),
                "yas_65_69": clean_numeric(values[3]),
                "yas_70_74": clean_numeric(values[4]),
                "yas_75_79": clean_numeric(values[5]),
                "yas_80_84": clean_numeric(values[6]),
                "yas_85_89": clean_numeric(values[7]),
                "yas_90_plus": clean_numeric(values[8]),
            })

    age = pd.DataFrame(rows)

    if age.empty:
        raise ValueError("Yaş dosyası parse edilemedi. Dosyanın | ayraçlı olup olmadığını kontrol et.")

    age = age.dropna(subset=["il_key", "yıl"]).copy()
    age["yıl"] = age["yıl"].astype(int)

    return age

df_age = parse_age_file(YAS_PATH)

print(df_age.shape)
display(df_age.head())
print("Yaş dosyası yıl aralığı:", df_age["yıl"].min(), "-", df_age["yıl"].max())
print("Yaş dosyası il sayısı:", df_age["il_key"].nunique())

(1539, 12)


,yıl,il_raw,il_key,yas_0_4,yas_10_14,yas_5_9,yas_65_69,yas_70_74,yas_75_79,yas_80_84,yas_85_89,yas_90_plus
0,2007,Adana-1,adana,170903.0,196529.0,192331.0,40794.0,30617.0,23068.0,12182.0,4004.0,2430.0
1,2007,Adıyaman-2,adiyaman,60457.0,69026.0,65992.0,11596.0,8838.0,7487.0,3828.0,1449.0,838.0
2,2007,Afyonkarahisar-3,afyonkarahisar,55658.0,61309.0,61660.0,21446.0,18284.0,14187.0,6239.0,1547.0,691.0
3,2007,Ağrı-4,agri,66964.0,70488.0,82191.0,6722.0,5069.0,3804.0,2067.0,827.0,318.0
4,2007,Aksaray-68,aksaray,32596.0,36850.0,37386.0,8575.0,6885.0,4498.0,2465.0,841.0,441.0


Yaş dosyası yıl aralığı: 2007 - 2025
Yaş dosyası il sayısı: 81


In [7]:
# ============================================================
# 6) ANA VERİ + YAŞ VERİSİ MERGE
# ============================================================

df = df_main.merge(
    df_age[["il_key", "yıl", "yas_0_4", "yas_5_9", "yas_10_14"]],
    on=["il_key", "yıl"],
    how="left"
)

# 0-14 yaş oranı: il bazlı genç nüfus / toplam il nüfusu * 100
df["yas_0_14"] = (
    df["yas_0_4"] + df["yas_5_9"] + df["yas_10_14"]
) / df["nufus"] * 100

# 100 bin kişi başına düşen toplam hastane yatak sayısı
# Eğer model_giris.csv içindeki yatak zaten toplam yatak sayısı ise doğru formül budur.
df["yatak_oran"] = df["yatak"] / df["nufus"] * 100000

print(df.shape)
display(
    df[[
        "il_adı", "yıl", "nufus", "yatak",
        "yas_0_4", "yas_5_9", "yas_10_14",
        "yas_0_14", "yatak_oran"
    ]].head(10)
)

print("\nyas_0_14 boş sayısı:", df["yas_0_14"].isna().sum())
print("yatak_oran boş sayısı:", df["yatak_oran"].isna().sum())


(1863, 17)


,il_adı,yıl,nufus,yatak,yas_0_4,yas_5_9,yas_10_14,yas_0_14,yatak_oran
0,Adana,2002,1916637.0,4266.0,NaN,NaN,NaN,NaN,222.577358
1,Adana,2003,1933428.0,4382.0,NaN,NaN,NaN,NaN,226.644075
2,Adana,2004,1951142.0,4207.0,NaN,NaN,NaN,NaN,215.617315
3,Adana,2005,1969512.0,4773.0,NaN,NaN,NaN,NaN,242.344296
4,Adana,2006,1988277.0,4973.0,NaN,NaN,NaN,NaN,250.116055
5,Adana,2007,2006650.0,5132.0,170903.0,192331.0,196529.0,27.895398,255.749632
6,Adana,2008,2026319.0,5146.0,179673.0,187993.0,196006.0,27.817535,253.958039
7,Adana,2009,2062226.0,5269.0,187143.0,183958.0,195251.0,27.463139,255.500610
8,Adana,2010,2085225.0,5918.0,186862.0,182019.0,197410.0,27.157309,283.806304
9,Adana,2011,2108805.0,6118.0,186032.0,181099.0,198518.0,26.823201,290.116915



yas_0_14 boş sayısı: 405
yatak_oran boş sayısı: 0


In [8]:
# ============================================================
# 7) HEDEF KONTROLÜ
# ============================================================

show_target_diagnostics(df.dropna(subset=["yas_0_14"]), "yas_0_14")
show_target_diagnostics(df.dropna(subset=["yatak_oran"]), "yatak_oran")



HEDEF KONTROLÜ: yas_0_14

Genel özet:
count    1458.000000
mean       23.774550
std         6.575846
min        13.926477
25%        19.045591
50%        21.857943
75%        26.933072
max        44.982825
Name: yas_0_14, dtype: float64

Yıl bazında varyans kontrolü:


,count,nunique,mean,std,min,max
yıl,,,,,,
2010,81,81,25.8949,7.0992,16.6671,44.8716
2011,81,81,25.4047,6.7563,15.0525,42.4929
2012,81,81,24.9456,6.7099,14.5347,41.9740
2013,81,81,24.5064,6.6704,14.9424,41.4864
2014,81,81,24.0612,6.5390,14.7942,40.7747
2015,81,81,23.7314,6.3901,14.5058,40.4047
2016,81,81,23.4034,6.2844,14.5061,40.1133
2017,81,81,23.1780,6.1581,14.3931,39.9902
2018,81,81,22.8619,6.0310,14.1420,39.8274



Sorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.

HEDEF KONTROLÜ: yatak_oran

Genel özet:
count    1863.000000
mean      255.554167
std        91.726671
min        39.711467
25%       195.061872
50%       247.727286
75%       302.805497
max       654.769062
Name: yatak_oran, dtype: float64

Yıl bazında varyans kontrolü:


,count,nunique,mean,std,min,max
yıl,,,,,,
2010,81,81,243.6028,87.9530,83.1312,482.8035
2011,81,81,249.2126,90.8562,92.9401,545.6601
2012,81,81,254.7575,87.3884,124.8442,503.0596
2013,81,81,254.4456,83.2912,124.1443,498.7725
2014,81,81,260.7155,89.7616,129.0247,641.1764
2015,81,81,264.3464,89.0440,126.0371,654.7691
2016,81,81,268.5514,80.6494,130.4290,509.8434
2017,81,81,274.8001,81.9887,125.3885,484.1944
2018,81,81,276.2323,82.3446,120.3762,502.1506



Sorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.


In [9]:
# ============================================================
# 8) FEATURE ENGINEERING
# ============================================================

targets = ["yas_0_14", "yatak_oran"]

df = df.sort_values(["il_key", "yıl"]).reset_index(drop=True)

# Hedefleri ve yaş bileşenlerini feature listesinden çıkarıyoruz.
# yatak_oran hedefi yatak/nufus ile hesaplandığı için "yatak" sütununu da feature olarak kullanmıyoruz.
# Aksi halde veri sızıntısı olur.
base_numeric_cols = [
    c for c in df.columns
    if c not in [
        "il_adı", "il_key", "il_raw",
        "yas_0_14", "yatak_oran",
        "yas_0_4", "yas_5_9", "yas_10_14",
        "yatak"
    ]
    and pd.api.types.is_numeric_dtype(df[c])
]

print("Başlangıç numeric feature sayısı:", len(base_numeric_cols))
print(base_numeric_cols)

for target in targets:
    df[f"{target}_lag1"] = df.groupby("il_key")[target].shift(1)
    df[f"{target}_lag2"] = df.groupby("il_key")[target].shift(2)
    df[f"{target}_diff1"] = df.groupby("il_key")[target].diff(1)
    df[f"{target}_roll3_mean"] = (
        df.groupby("il_key")[target]
        .shift(1)
        .rolling(3)
        .mean()
        .reset_index(level=0, drop=True)
    )

feature_cols = [
    c for c in df.columns
    if c not in ["il_adı", "il_key", "il_raw", "yatak"] + targets
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nToplam feature sayısı:", len(feature_cols))
display(df[["il_adı", "yıl"] + targets + [f"{t}_lag1" for t in targets]].head(15))


Başlangıç numeric feature sayısı: 9
['yıl', 'hastane', 'nufus', 'asistan', 'diger', 'ebe', 'hemsire', 'pratisyen', 'uzman']

Toplam feature sayısı: 20


,il_adı,yıl,yas_0_14,yatak_oran,yas_0_14_lag1,yatak_oran_lag1
0,Adana,2002,NaN,222.577358,NaN,NaN
1,Adana,2003,NaN,226.644075,NaN,222.577358
2,Adana,2004,NaN,215.617315,NaN,226.644075
3,Adana,2005,NaN,242.344296,NaN,215.617315
4,Adana,2006,NaN,250.116055,NaN,242.344296
5,Adana,2007,27.895398,255.749632,NaN,250.116055
6,Adana,2008,27.817535,253.958039,27.895398,255.749632
7,Adana,2009,27.463139,255.500610,27.817535,253.958039
8,Adana,2010,27.157309,283.806304,27.463139,255.500610
9,Adana,2011,26.823201,290.116915,27.157309,283.806304


In [10]:
# ============================================================
# 9) MODELLER
# ============================================================

def make_models():
    models = {}

    models["LinearRegression"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    models["Ridge"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    models["ElasticNet"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=10000))
    ])

    models["KNN"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ])

    models["SVR"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR(C=10, epsilon=0.1))
    ])

    models["RandomForest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=2
        ))
    ])

    models["ExtraTrees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=2
        ))
    ])

    models["GradientBoosting"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(random_state=42))
    ])

    models["HistGradientBoosting"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    models["Bagging"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", BaggingRegressor(
            estimator=RandomForestRegressor(n_estimators=100, random_state=42),
            n_estimators=10,
            random_state=42,
            n_jobs=-1
        ))
    ])

    if HAS_XGB:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                objective="reg:squarederror"
            ))
        ])

    if HAS_CAT:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=300,
                learning_rate=0.05,
                depth=4,
                random_seed=42,
                verbose=False
            ))
        ])

    voting_estimators = [
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ("et", ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ("gb", GradientBoostingRegressor(random_state=42))
    ]

    models["Voting"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", VotingRegressor(voting_estimators))
    ])

    stacking_estimators = [
        ("ridge", Ridge(alpha=1.0)),
        ("rf", RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
        ("gb", GradientBoostingRegressor(random_state=42))
    ]

    models["Stacking"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", StackingRegressor(
            estimators=stacking_estimators,
            final_estimator=Ridge(alpha=1.0),
            n_jobs=-1
        ))
    ])

    return models

models = make_models()
print("Model listesi:")
print(list(models.keys()))

Model listesi:
['LinearRegression', 'Ridge', 'ElasticNet', 'KNN', 'SVR', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'HistGradientBoosting', 'Bagging', 'XGBoost', 'CatBoost', 'Voting', 'Stacking']


In [11]:
# ============================================================
# 10) ZAMAN BAZLI TEST
# ============================================================

def train_eval_ml_models(df, target, test_years=None):
    if test_years is None:
        years = sorted(df["yıl"].dropna().unique())
        test_years = years[-3:]

    rows = []
    preds = []

    for test_year in test_years:
        train_df = df[(df["yıl"] < test_year) & df[target].notna()].copy()
        test_df = df[(df["yıl"] == test_year) & df[target].notna()].copy()

        if len(train_df) < 30 or len(test_df) == 0:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target]
        X_test = test_df[feature_cols]
        y_test = test_df[target]

        # Baseline: lag1
        if f"{target}_lag1" in test_df.columns:
            y_base = test_df[f"{target}_lag1"].values
            mask = ~pd.isna(y_base)
            if mask.sum() > 0:
                met = calc_regression_metrics(y_test.values[mask], y_base[mask])
                rows.append({
                    "Hedef": target,
                    "Model": "Lag1_Baseline",
                    "TestYili": test_year,
                    **met
                })

                tmp = test_df.loc[mask, ["il_adı", "yıl"]].copy()
                tmp["Hedef"] = target
                tmp["Model"] = "Lag1_Baseline"
                tmp["y_true"] = y_test.values[mask]
                tmp["y_pred"] = y_base[mask]
                preds.append(tmp)

        for name, model in models.items():
            try:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                met = calc_regression_metrics(y_test, y_pred)
                rows.append({
                    "Hedef": target,
                    "Model": name,
                    "TestYili": test_year,
                    **met
                })

                tmp = test_df[["il_adı", "yıl"]].copy()
                tmp["Hedef"] = target
                tmp["Model"] = name
                tmp["y_true"] = y_test.values
                tmp["y_pred"] = y_pred
                preds.append(tmp)

            except Exception as e:
                print(f"Hata | Target={target} | Model={name} | TestYili={test_year} | {e}")

    metrics = pd.DataFrame(rows)
    pred_df = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()

    return metrics, pred_df

all_metrics = []
all_preds = []

for target in targets:
    print("\n" + "="*100)
    print("EĞİTİM BAŞLADI:", target)
    print("="*100)

    m, p = train_eval_ml_models(df, target)
    all_metrics.append(m)
    all_preds.append(p)

metrics_all = pd.concat(all_metrics, ignore_index=True)
preds_all = pd.concat(all_preds, ignore_index=True)

print("\nMetrikler:")
display(metrics_all.head())

print("\nTahminler:")
display(preds_all.head())


EĞİTİM BAŞLADI: yas_0_14

EĞİTİM BAŞLADI: yatak_oran

Metrikler:


,Hedef,Model,TestYili,Accuracy,MAE,RMSE,R2,ErrorRate
0,yas_0_14,Lag1_Baseline,2022,97.859266,0.452396,0.479494,0.992630,2.140734
1,yas_0_14,LinearRegression,2022,96.319877,0.800959,1.242620,0.950501,3.680123
2,yas_0_14,Ridge,2022,96.489831,0.745799,0.991487,0.968487,3.510169
3,yas_0_14,ElasticNet,2022,96.564657,0.718328,0.876136,0.975393,3.435343
4,yas_0_14,KNN,2022,96.267279,0.751412,1.062453,0.963814,3.732721



Tahminler:


,il_adı,yıl,Hedef,Model,y_true,y_pred
0,Adana,2022,yas_0_14,Lag1_Baseline,23.712483,24.143877
1,Adıyaman,2022,yas_0_14,Lag1_Baseline,27.963739,28.352380
2,Afyonkarahisar,2022,yas_0_14,Lag1_Baseline,20.826829,21.298774
3,Ağrı,2022,yas_0_14,Lag1_Baseline,32.980890,33.690464
4,Aksaray,2022,yas_0_14,Lag1_Baseline,23.603930,24.071420


In [12]:
# ============================================================
# 11) MODEL KARŞILAŞTIRMA
# ============================================================

summary = (
    metrics_all
    .groupby(["Hedef", "Model"], as_index=False)
    .agg({
        "Accuracy": "mean",
        "MAE": "mean",
        "RMSE": "mean",
        "R2": "mean",
        "ErrorRate": "mean"
    })
)

summary = summary.sort_values(
    ["Hedef", "R2", "RMSE", "MAE", "ErrorRate"],
    ascending=[True, False, True, True, True]
)

print("\n========== HEDEF × MODEL ÖZETİ ==========")
display(summary.round(4))

best_models = (
    summary
    .sort_values(["Hedef", "R2", "RMSE", "MAE", "ErrorRate"],
                 ascending=[True, False, True, True, True])
    .groupby("Hedef", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("\n========== HER HEDEF İÇİN EN İYİ MODEL ==========")
display(best_models.round(4))


========== HEDEF × MODEL ÖZETİ ==========


,Hedef,Model,Accuracy,MAE,RMSE,R2,ErrorRate
5,yas_0_14,HistGradientBoosting,99.3851,0.1276,0.1909,0.9988,0.6149
3,yas_0_14,ExtraTrees,99.3537,0.1304,0.1849,0.9988,0.6463
9,yas_0_14,RandomForest,99.3436,0.1345,0.1950,0.9986,0.6564
13,yas_0_14,Voting,99.2626,0.1481,0.1984,0.9986,0.7374
12,yas_0_14,Stacking,99.3063,0.1454,0.2051,0.9986,0.6937
0,yas_0_14,Bagging,99.1930,0.1660,0.2267,0.9982,0.8070
14,yas_0_14,XGBoost,99.0144,0.1931,0.2508,0.9979,0.9856
4,yas_0_14,GradientBoosting,98.8711,0.2218,0.2933,0.9971,1.1289
1,yas_0_14,CatBoost,98.4278,0.3103,0.3833,0.9950,1.5722
7,yas_0_14,Lag1_Baseline,97.7097,0.4790,0.5206,0.9907,2.2903



========== HER HEDEF İÇİN EN İYİ MODEL ==========


,Hedef,Model,Accuracy,MAE,RMSE,R2,ErrorRate
0,yas_0_14,HistGradientBoosting,99.3851,0.1276,0.1909,0.9988,0.6149
1,yatak_oran,ElasticNet,98.6953,3.5403,4.2802,0.9974,1.3047


In [13]:
# ============================================================
# 12) EN İYİ MODEL TAHMİNLERİNİ İNCELE
# ============================================================

for _, row in best_models.iterrows():
    target = row["Hedef"]
    model = row["Model"]

    print("\n" + "="*100)
    print(f"HEDEF: {target} | EN İYİ MODEL: {model}")
    print("="*100)

    pp = preds_all[
        (preds_all["Hedef"] == target) &
        (preds_all["Model"] == model)
    ].copy()

    pp["abs_error"] = np.abs(pp["y_true"] - pp["y_pred"])
    pp["pct_error"] = pp["abs_error"] / np.maximum(np.abs(pp["y_true"]), 1e-9) * 100

    display(
        pp[["il_adı", "yıl", "y_true", "y_pred", "abs_error", "pct_error"]]
        .sort_values(["yıl", "abs_error"], ascending=[True, False])
        .head(20)
        .round(4)
    )

    print("Hata özeti:")
    display(pp[["abs_error", "pct_error"]].describe().round(4))


HEDEF: yas_0_14 | EN İYİ MODEL: HistGradientBoosting


,il_adı,yıl,y_true,y_pred,abs_error,pct_error
764,Hakkari,2022,27.7515,26.8166,0.9349,3.3689
776,Kilis,2022,27.6374,27.1717,0.4657,1.6851
804,Tunceli,2022,14.5509,14.8653,0.3144,2.1607
756,Edirne,2022,14.6154,14.9214,0.3060,2.0937
775,Kayseri,2022,22.6315,22.9368,0.3054,1.3492
761,Gaziantep,2022,30.8007,31.0924,0.2917,0.9471
790,Niğde,2022,22.1559,21.8769,0.2791,1.2595
773,Kars,2022,25.2575,24.9855,0.2720,1.0771
732,Ağrı,2022,32.9809,32.7199,0.2610,0.7913
763,Gümüşhane,2022,16.7105,16.9648,0.2543,1.5217


Hata özeti:


,abs_error,pct_error
count,243.0000,243.0000
mean,0.1276,0.6149
std,0.1427,0.6960
min,0.0015,0.0078
25%,0.0328,0.1736
50%,0.0778,0.3762
75%,0.1755,0.8242
max,0.9349,5.0736



HEDEF: yatak_oran | EN İYİ MODEL: ElasticNet


,il_adı,yıl,y_true,y_pred,abs_error,pct_error
3894,Ankara,2022,413.0201,432.0499,19.0298,4.6075
3902,Bayburt,2022,379.8625,369.1368,10.7258,2.8236
3940,Konya,2022,382.6512,391.9849,9.3337,2.4392
3933,Kastamonu,2022,308.6363,317.2506,8.6143,2.7911
3899,Balıkesir,2022,269.1656,276.5206,7.3550,2.7325
3912,Denizli,2022,300.3790,307.1479,6.7688,2.2534
3946,Muğla,2022,223.2430,229.6642,6.4212,2.8763
3895,Antalya,2022,282.4029,288.7240,6.3212,2.2383
3890,Afyonkarahisar,2022,313.6893,320.0000,6.3107,2.0118
3921,Giresun,2022,387.9236,394.1978,6.2742,1.6174


Hata özeti:


,abs_error,pct_error
count,243.0000,243.0000
mean,3.5403,1.3047
std,2.4217,0.9240
min,0.0106,0.0041
25%,1.9581,0.5986
50%,3.4965,1.1900
75%,4.6461,1.8965
max,19.0298,4.8000


In [14]:
# ============================================================
# 13) SONUÇLARI KAYDET
# ============================================================

summary.to_csv("yas_0_14_yatak_oran_model_ozet.csv", index=False, encoding="utf-8-sig")
best_models.to_csv("yas_0_14_yatak_oran_en_iyi_modeller.csv", index=False, encoding="utf-8-sig")
preds_all.to_csv("yas_0_14_yatak_oran_tahminler.csv", index=False, encoding="utf-8-sig")

print("Kaydedildi:")
print("- yas_0_14_yatak_oran_model_ozet.csv")
print("- yas_0_14_yatak_oran_en_iyi_modeller.csv")
print("- yas_0_14_yatak_oran_tahminler.csv")


Kaydedildi:
- yas_0_14_yatak_oran_model_ozet.csv
- yas_0_14_yatak_oran_en_iyi_modeller.csv
- yas_0_14_yatak_oran_tahminler.csv
